# Phase 11.1 — Recommendation Contract

Contracts only: recommendation intent, candidates, and responses (no recommendation engine).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "productiq").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))

from productiq.recommendation.config import RecommendationConfig
from productiq.recommendation.contracts import (
    IMPLEMENTED_RECOMMENDATION_TYPES,
    RankedRecommendation,
    RecommendationCandidate,
    RecommendationCandidateSource,
    RecommendationRequest,
    RecommendationResponse,
    RecommendationType,
    SUPPORTED_RECOMMENDATION_TYPES,
)
from productiq.recommendation.invariants import (
    deterministic_recommendation_sort_key,
    validate_recommendation_response_invariants,
    validate_recommendation_type_implemented,
)
from productiq.exceptions import RecommendationError

In [ ]:
list(RecommendationType), SUPPORTED_RECOMMENDATION_TYPES, IMPLEMENTED_RECOMMENDATION_TYPES

In [ ]:
config = RecommendationConfig()
request = RecommendationRequest(
    seed_product_id="P123",
    recommendation_type=RecommendationType.SIMILAR,
    top_k=10,
    config=config,
)
request.model_dump(mode="json")

In [ ]:
candidate = RecommendationCandidate(
    product_id="P200",
    candidate_generation_score=0.42,
    source=RecommendationCandidateSource.VECTOR,
)
candidate.model_dump(mode="json")

In [ ]:
candidate_a = RecommendationCandidate(
    product_id="A",
    candidate_generation_score=0.42,
    source=RecommendationCandidateSource.VECTOR,
)
rows = (
    RankedRecommendation(
        product_id="A",
        rank=1,
        recommendation_score=0.5,
        candidate=candidate_a,
    ),
    RankedRecommendation(
        product_id="B",
        rank=2,
        recommendation_score=0.5,
        candidate=RecommendationCandidate(
            product_id="B",
            candidate_generation_score=0.4,
            source=RecommendationCandidateSource.ATTRIBUTE,
        ),
    ),
)
response = RecommendationResponse(
    seed_product_id="P123",
    recommendation_type=RecommendationType.SIMILAR,
    recommendations=rows,
    requested_top_k=10,
    config=config,
)
validate_recommendation_response_invariants(response)
response.model_dump(mode="json")

In [ ]:
deterministic_recommendation_sort_key(
    recommendation_score=0.5,
    product_id="A",
    config=config,
)
< deterministic_recommendation_sort_key(
    recommendation_score=0.5,
    product_id="B",
    config=config,
)

In [ ]:
try:
    RecommendationResponse(
        seed_product_id="P123",
        recommendation_type=RecommendationType.SIMILAR,
        recommendations=(
            RankedRecommendation(
                product_id="P123",
                rank=1,
                recommendation_score=0.9,
                candidate=None,
            ),
        ),
        requested_top_k=5,
        config=config,
    )
except Exception as exc:
    type(exc).__name__, str(exc)

In [ ]:
try:
    validate_recommendation_type_implemented(RecommendationType.SIMILAR)
except RecommendationError as exc:
    str(exc)